In [1]:
!pip install numpy==1.26.4 pandas==2.2.2 --force-reinstall --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 336.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 227.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 353.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 376.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.5/348.5 kB 246.2 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2026.1.post1
    Uninstalling pytz-2026.1.post1:
      Successfully uninstalled pytz-2026.1.post1
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2025.3
    Uninstalling tzdata-2025.3:
      Successfully uninstalled tzdata-2025.3
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: numpy
    Foun

In [2]:
# Check GPU
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Training will be slow.")
    print("Enable GPU: Settings → Accelerator → GPU T4 x2")

# Option 1: If you uploaded your own files
# Adjust these paths based on where you uploaded the files
TRAIN_PATH = "/kaggle/input/datasets/kritikgianta/comment-dataset/train.csv"  # Update this path
TEST_PATH = "/kaggle/input/datasets/kritikgianta/comment-dataset/test.csv"    # Update this path

# Option 2: Use Kaggle's Jigsaw Toxic Comment Classification dataset
# Uncomment these lines if using Kaggle dataset:
# TRAIN_PATH = "/kaggle/input/jigsaw-toxic-comment-classification-challenge/train.csv"
# TEST_PATH = "/kaggle/input/jigsaw-toxic-comment-classification-challenge/test.csv"

import os
print(f"Train file exists: {os.path.exists(TRAIN_PATH)}")
print(f"Test file exists: {os.path.exists(TEST_PATH)}")

# Imports
import numpy as np
import pandas as pd
import torch
import json
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

# Disable warnings
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

print("✅ All imports successful")

# Training configuration
config = {
    "model": {
        "name": "distilbert-base-uncased",
        "max_length": 128,
        "num_labels": 1
    },
    "training": {
        "epochs": 2,
        "batch_size": 32,  # T4 GPU can handle this
        "learning_rate": 3e-5,
        "weight_decay": 0.01,
        "warmup_steps": 500,
        "logging_steps": 100,
        "eval_steps": 500,
        "save_steps": 1000,
        "fp16": True  # Faster on T4
    },
    "data": {
        "validation_split": 0.1,
        "random_seed": 42
    },
    "output": {
        "model_dir": "/kaggle/working/exported_model"
    }
}

print("Configuration:")
print(json.dumps(config, indent=2))

print("📊 Loading training data...")
train_df = pd.read_csv(TRAIN_PATH)

print(f"Shape: {train_df.shape}")
print(f"Columns: {train_df.columns.tolist()}")
print("\nFirst few rows:")
display(train_df.head())

# Check required columns
assert "comment_text" in train_df.columns, "Missing 'comment_text' column!"
assert "toxic" in train_df.columns, "Missing 'toxic' column!"

print(f"\n📈 Statistics:")
print(f"Total comments: {len(train_df):,}")
print(f"Toxic comments: {train_df['toxic'].sum():,}")
print(f"Non-toxic comments: {(len(train_df) - train_df['toxic'].sum()):,}")
print(f"Toxicity rate: {train_df['toxic'].mean():.2%}")

# Clean and prepare data
print("🧹 Preparing data...")
train_df = train_df[["comment_text", "toxic"]].copy()
train_df["toxic"] = train_df["toxic"].astype(float)

# Remove NaN
initial_len = len(train_df)
train_df = train_df.dropna()
if initial_len > len(train_df):
    print(f"Removed {initial_len - len(train_df)} rows with missing values")

# Train/val split
dataset = Dataset.from_pandas(train_df)
dataset = dataset.train_test_split(
    test_size=config["data"]["validation_split"],
    seed=config["data"]["random_seed"]
)

train_ds = dataset["train"]
val_ds = dataset["test"]

print(f"\n✅ Split complete:")
print(f"   Training: {len(train_ds):,} samples")
print(f"   Validation: {len(val_ds):,} samples")

MODEL_NAME = config["model"]["name"]

print(f"🔤 Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=config["model"]["max_length"]
    )

print("🔄 Tokenizing... (this takes a few minutes)")
train_ds = train_ds.map(preprocess, batched=True, desc="Tokenizing train")
val_ds = val_ds.map(preprocess, batched=True, desc="Tokenizing validation")

# Rename and format
train_ds = train_ds.rename_column("toxic", "labels")
val_ds = val_ds.rename_column("toxic", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("✅ Tokenization complete")

print("🤖 Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=config["model"]["num_labels"]
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"✅ Model loaded on: {device}")

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.squeeze(logits)
    labels = labels.astype(int)
    probs = 1 / (1 + np.exp(-logits))
    
    return {
        "roc_auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs),
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="/kaggle/working/toxicity_model",
    per_device_train_batch_size=config["training"]["batch_size"],
    per_device_eval_batch_size=config["training"]["batch_size"],
    num_train_epochs=config["training"]["epochs"],
    learning_rate=config["training"]["learning_rate"],
    weight_decay=config["training"]["weight_decay"],
    warmup_steps=config["training"]["warmup_steps"],
    logging_steps=config["training"]["logging_steps"],
    eval_steps=config["training"]["eval_steps"],
    save_steps=config["training"]["save_steps"],
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    fp16=config["training"]["fp16"],
    report_to="none",
    dataloader_pin_memory=True,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("✅ Trainer configured")
# Train!
print("🚀 Starting training...")
print(f"   Epochs: {config['training']['epochs']}")
print(f"   Batch size: {config['training']['batch_size']}")
print(f"   Device: {device}")
print("\n⏳ Estimated time: 15-20 minutes on T4 GPU\n")

trainer.train()

print("\n✅ Training complete!")
print("📊 Evaluating model...")
val_metrics = trainer.evaluate()

print("\n✅ Validation Metrics:")
for key, value in val_metrics.items():
    if key.startswith("eval_"):
        print(f"   {key.replace('eval_', '').upper()}: {value:.4f}")

# Get predictions
print("🔮 Generating predictions...")
preds = trainer.predict(val_ds)
logits = np.squeeze(preds.predictions)
labels = preds.label_ids.astype(int)
probs = 1 / (1 + np.exp(-logits))

print(f"✅ Generated {len(probs):,} predictions")
# Find optimal threshold
print("🎯 Finding optimal threshold...")

thresholds = np.linspace(0.1, 0.9, 81)
f1_scores = []
precision_scores = []
recall_scores = []

for t in thresholds:
    predictions = (probs >= t).astype(int)
    f1 = f1_score(labels, predictions)
    f1_scores.append(f1)
    
    prec = precision_score(labels, predictions, zero_division=0)
    rec = recall_score(labels, predictions, zero_division=0)
    precision_scores.append(prec)
    recall_scores.append(rec)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"\n✅ Optimal Threshold: {best_threshold:.3f}")
print(f"✅ Best F1 Score: {best_f1:.4f}")
# Confusion matrix
optimal_preds = (probs >= best_threshold).astype(int)
cm = confusion_matrix(labels, optimal_preds)

print("\n📊 Confusion Matrix @ Optimal Threshold:")
print(f"              Predicted")
print(f"            Non-toxic  Toxic")
print(f"Actual")
print(f"Non-toxic    {cm[0][0]:6d}  {cm[0][1]:6d}")
print(f"Toxic        {cm[1][0]:6d}  {cm[1][1]:6d}")

print("\n📋 Classification Report:")
print(classification_report(labels, optimal_preds, target_names=['Non-toxic', 'Toxic']))
SAVE_DIR = config["output"]["model_dir"]

print(f"💾 Saving model to {SAVE_DIR}...")

# Save model and tokenizer
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save threshold info
threshold_info = {
    "optimal_threshold": float(best_threshold),
    "f1_score": float(best_f1),
    "roc_auc": float(val_metrics["eval_roc_auc"]),
    "pr_auc": float(val_metrics["eval_pr_auc"])
}

with open(f"{SAVE_DIR}/threshold_info.json", "w") as f:
    json.dump(threshold_info, f, indent=2)

# Save training config
with open(f"{SAVE_DIR}/training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Model saved!")
print(f"\n📂 Files in {SAVE_DIR}:")
!ls -lh {SAVE_DIR}
print("🧪 Testing model with sample comments:\n")

test_samples = [
    "You are a stupid idiot",
    "Thank you so much for your help",
    "Go die, nobody likes you",
    "This is a wonderful community",
    "Fuck you asshole",
    "I really appreciate your thoughtful response",
    "You're pathetic and useless",
    "Have a great day!"
]

print("=" * 70)
for sample in test_samples:
    inputs = tokenizer(sample, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        logit = model(**inputs).logits.item()
        prob = 1 / (1 + torch.exp(-torch.tensor(logit))).item()
    
    label = "Toxic" if prob >= best_threshold else "Non-toxic"
    emoji = "🔴" if prob >= best_threshold else "🟢"
    
    print(f"{emoji} {prob:.3f} | {label:12s} | {sample}")

print("=" * 70)
print("\n" + "=" * 80)
print("✅ TRAINING COMPLETE!")
print("=" * 80)
print(f"\n📌 Model saved to: {SAVE_DIR}")
print(f"📌 Optimal threshold: {best_threshold:.3f}")
print(f"📌 F1 Score: {best_f1:.4f}")
print(f"📌 ROC-AUC: {val_metrics['eval_roc_auc']:.4f}")
print("\n📥 TO DOWNLOAD:")
print("   1. Click the folder icon on the right →")
print("   2. Navigate to /kaggle/working/exported_model/")
print("   3. Click the three dots ⋮ next to 'exported_model'")
print("   4. Select 'Download'")
print("   5. Extract the zip file")
print("   6. Place the 'exported_model' folder in your project directory")
print("\n🚀 Then run: streamlit run app.py")
print("=" * 80)

CUDA Available: True
GPU: Tesla T4
Memory: 14.6 GB
Train file exists: True
Test file exists: True
✅ All imports successful
Configuration:
{
  "model": {
    "name": "distilbert-base-uncased",
    "max_length": 128,
    "num_labels": 1
  },
  "training": {
    "epochs": 2,
    "batch_size": 32,
    "learning_rate": 3e-05,
    "weight_decay": 0.01,
    "warmup_steps": 500,
    "logging_steps": 100,
    "eval_steps": 500,
    "save_steps": 1000,
    "fp16": true
  },
  "data": {
    "validation_split": 0.1,
    "random_seed": 42
  },
  "output": {
    "model_dir": "/kaggle/working/exported_model"
  }
}
📊 Loading training data...
Shape: (159571, 8)
Columns: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

First few rows:


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0



📈 Statistics:
Total comments: 159,571
Toxic comments: 15,294
Non-toxic comments: 144,277
Toxicity rate: 9.58%
🧹 Preparing data...

✅ Split complete:
   Training: 143,613 samples
   Validation: 15,958 samples
🔤 Loading tokenizer: distilbert-base-uncased


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...


✅ Validation Metrics:
   LOSS: 0.0475
   ROC_AUC: 0.9856
   PR_AUC: 0.9120
   RUNTIME: 33.9899
   SAMPLES_PER_SECOND: 469.4920
   STEPS_PER_SECOND: 7.3550
🔮 Generating predictions...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


✅ Generated 15,958 predictions
🎯 Finding optimal threshold...

✅ Optimal Threshold: 0.610
✅ Best F1 Score: 0.8388

📊 Confusion Matrix @ Optimal Threshold:
              Predicted
            Non-toxic  Toxic
Actual
Non-toxic     14200     265
Toxic           223    1270

📋 Classification Report:
              precision    recall  f1-score   support

   Non-toxic       0.98      0.98      0.98     14465
       Toxic       0.83      0.85      0.84      1493

    accuracy                           0.97     15958
   macro avg       0.91      0.92      0.91     15958
weighted avg       0.97      0.97      0.97     15958

💾 Saving model to /kaggle/working/exported_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved!

📂 Files in /kaggle/working/exported_model:
total 257M
-rw-r--r-- 1 root root  724 Mar 20 15:17 config.json
-rw-r--r-- 1 root root 256M Mar 20 15:17 model.safetensors
-rw-r--r-- 1 root root  129 Mar 20 15:17 threshold_info.json
-rw-r--r-- 1 root root  322 Mar 20 15:17 tokenizer_config.json
-rw-r--r-- 1 root root 695K Mar 20 15:17 tokenizer.json
-rw-r--r-- 1 root root 5.1K Mar 20 15:17 training_args.bin
-rw-r--r-- 1 root root  468 Mar 20 15:17 training_config.json
🧪 Testing model with sample comments:

🔴 0.736 | Toxic        | You are a stupid idiot
🟢 0.502 | Non-toxic    | Thank you so much for your help
🔴 0.735 | Toxic        | Go die, nobody likes you
🟢 0.504 | Non-toxic    | This is a wonderful community
🔴 0.738 | Toxic        | Fuck you asshole
🟢 0.503 | Non-toxic    | I really appreciate your thoughtful response
🔴 0.734 | Toxic        | You're pathetic and useless
🟢 0.504 | Non-toxic    | Have a great day!

✅ TRAINING COMPLETE!

📌 Model saved to: /kaggle/working/exp

In [3]:
# Optional: Create a zip file for easier download
import shutil

print("📦 Creating zip file for download...")
shutil.make_archive(
    "/kaggle/working/exported_model_archive",
    "zip",
    "/kaggle/working/",
    "exported_model"
)
print("✅ Created: /kaggle/working/exported_model_archive.zip")
print("\nYou can download this single file instead of the folder!")

📦 Creating zip file for download...
✅ Created: /kaggle/working/exported_model_archive.zip

You can download this single file instead of the folder!
